# Step 4 — TOD Zone Buffer Generation

Reads the finalized `tod_access_points` layer from the shared GeoPackage and
generates simple full-circle Euclidean buffers at **200 ft, ¼ mile, and ½ mile**
around each pedestrian access point. Each buffer is tagged with `tod_tier` and
`buffer_band` for downstream processing.

Overlap resolution, tier precedence (Tier 1 supersedes Tier 2), jurisdictional
population thresholds, and ring differencing are **not** applied here — these
relationships are resolved in a downstream step using `tod_tier`, `buffer_band`,
and geometry.

> **Pipeline order:** Run after `3_tod_station_assignment_reintegration.ipynb`.
> Requires the `tod_access_points` and `tod_stations` layers to be finalized
> in the shared GeoPackage before proceeding.
> All data source paths are defined in `config.py`.


In [1]:
import uuid
import pandas as pd
import geopandas as gpd
from mtcpy.census import pull_acs_data
from mtcpy.geospatial import repair_geometry
from config import *

Info: Found credentials at: /Users/jcroff/Library/CloudStorage/Box-Box/dvutils-creds-jcroff.json


## Inputs

In [2]:
# Load Bay Area jurisdiction boundaries from the MTC ArcGIS REST service.
# Returns incorporated places and unincorporated county lands (polygon layer).
jurisdictions_gdf = gpd.read_file(JURISDICTION_BOUNDARIES_URL)
print(f"Loaded {len(jurisdictions_gdf):,} jurisdiction features")
print("CRS:", jurisdictions_gdf.crs)

# Load finalized TOD layers from the shared GeoPackage.
tod_stations_gdf = gpd.read_file(TOD_DATABASE_GPKG, layer=GPKG_TOD_STATIONS_FINAL_LAYER)
print(f"Loaded {len(tod_stations_gdf):,} TOD stations")

tod_stops_gdf = gpd.read_file(TOD_DATABASE_GPKG, layer=GPKG_TOD_STOPS_FINAL_LAYER)
print(f"Loaded {len(tod_stops_gdf):,} TOD stops")

tod_access_pts_gdf = gpd.read_file(TOD_DATABASE_GPKG, layer=GPKG_TOD_ACCESS_PTS_FINAL_LAYER)
print(f"Loaded {len(tod_access_pts_gdf):,} TOD access points")

Loaded 109 jurisdiction features
CRS: EPSG:4326
Loaded 432 TOD stations
Loaded 769 TOD stops
Loaded 1,159 TOD access points


In [3]:
# Pull ACS 5-year (2019–2023) Total Population estimates at the Census place
# geography level via mtcpy.census.pull_acs_data.
#
# Parameters come from config.py:
#   ACS_YEAR = 2023        — endpoint year of the 5-year window
#   ACS_TYPE = "acs5"      — 5-year ACS product
#   ACS_TABLE_ID = "B01003" — Total Population table
#   ACS_GEOGRAPHY_LEVEL = "place" — incorporated places / CDPs
#
# The returned DataFrame includes:
#   B01003_001E  — total population estimate
#   NAME         — e.g. "San Jose city, California"
#   state        — FIPS state code ("06" for California)
#   place        — FIPS place code (5-digit string)
pop_df = pull_acs_data(
    year=ACS_YEAR,
    acs_type=ACS_TYPE,
    table_id=ACS_TABLE_ID,
    geography_level=ACS_GEOGRAPHY_LEVEL,
)

# Rename the estimate column for clarity and cast to integer.
pop_df = pop_df.rename(columns={"B01003_001E": "total_population"})
pop_df["total_population"] = pd.to_numeric(pop_df["total_population"], errors="coerce")

print(f"Pulled {len(pop_df):,} ACS place records")
pop_df[["jurisdiction", "total_population"]].head(5)

Pulled 101 ACS place records


,jurisdiction,total_population
0,Alameda,76876
1,Albany,19768
2,American Canyon,21698
3,Antioch,115759
4,Atherton,7021


In [4]:
# Join ACS population estimates to jurisdiction boundaries.
# Rename name columns to a shared "jurisdiction" key for consistency, then
# fix the one spelling mismatch before joining.

jurisdictions_gdf = jurisdictions_gdf.rename(columns={"jurname": "jurisdiction"})

jurisdictions_gdf["jurisdiction"] = jurisdictions_gdf["jurisdiction"].replace(
    "Saint Helena", "St. Helena"
)

jurisdictions_with_pop = jurisdictions_gdf.merge(
    pop_df[["jurisdiction", "total_population"]],
    on="jurisdiction",
    how="left",
)

jurisdictions_with_pop


,objectid,fipst,fipco,jurisdiction,Shape__Area,Shape__Length,geometry,total_population
0,219,06,097,Unincorporated Sonoma,0.450046,8.231250,"MULTIPOLYGON (((-122.62741 38.66751, -122.6238...",NaN
1,220,06,041,Unincorporated Marin,0.196976,5.260678,"MULTIPOLYGON (((-122.34747 38.07327, -122.3516...",NaN
2,221,06,055,Unincorporated Napa,0.201949,4.566464,"MULTIPOLYGON (((-122.10329 38.51335, -122.1033...",NaN
3,222,06,095,Unincorporated Solano,0.200921,6.225325,"MULTIPOLYGON (((-121.66836 38.17504, -121.6708...",NaN
4,223,06,013,Unincorporated Contra Costa,0.128281,8.168448,"MULTIPOLYGON (((-121.55701 37.81649, -121.5578...",NaN
...,...,...,...,...,...,...,...,...
104,323,06,081,San Mateo,0.004186,0.465963,"POLYGON ((-122.27668 37.53423, -122.27673 37.5...",103555.0
105,324,06,081,South San Francisco,0.007979,0.701870,"POLYGON ((-122.39193 37.67177, -122.39134 37.6...",64487.0
106,325,06,081,Woodside,0.003023,0.481513,"POLYGON ((-122.27071 37.45591, -122.2711 37.45...",5181.0
107,326,06,013,El Cerrito,0.000978,0.164531,"POLYGON ((-122.31972 37.93834, -122.31957 37.9...",25781.0


In [5]:
# Join access points → stations → stops.
#
# Relationship summary:
#   access_points → stations : many-to-one  (many access pts share one station)
#   stations      → stops    : one-to-many  (one station serves many stops)
#
# First join: left join access points to stations so every access point is
# retained; unmatched station_ids surface in the indicator check below.
stations_req_cols = ["station_id", "station_name"]
access_req_cols = ["access_id", "station_id", "access_point_name", "geometry"]
stops_req_cols = [
    "stop_id",
    "station_id",
    "stop_name",
    "route_short_name",
    "route_type",
    "tod_tier",
]

tod_access_stations_gdf = pd.merge(
    tod_access_pts_gdf[access_req_cols],
    tod_stations_gdf[stations_req_cols],
    on="station_id",
    how="left",
    indicator=True,
)

# Check: access points with a non-null station_id that didn't match a station record.
print("Access points with unmatched station_id:")
display(
    tod_access_stations_gdf[
        (tod_access_stations_gdf["_merge"] != "both")
        & (tod_access_stations_gdf["station_id"].notnull())
    ]
)

# Second join: OUTER join so records from both sides are retained.
# This lets us catch both directions of mismatch:
#   left_only  → access points with no matching stop (by station_id)
#   right_only → stops with no matching access point chain
tod_merged_gdf = pd.merge(
    tod_access_stations_gdf.drop(columns=["_merge"]),
    tod_stops_gdf[stops_req_cols],
    on="station_id",
    how="outer",
    indicator=True,
)

# Detect directional gaps.
stops_no_access = tod_merged_gdf[
    (tod_merged_gdf["_merge"] == "right_only")
    & (tod_merged_gdf["station_id"].notnull())
]
access_no_stops = tod_merged_gdf[
    (tod_merged_gdf["_merge"] == "left_only")
    & (tod_merged_gdf["station_id"].notnull())
]

# If any gaps exist, export a review workbook and raise so execution stops.
if len(stops_no_access) > 0 or len(access_no_stops) > 0:
    with pd.ExcelWriter(LINKAGE_GAPS_XLSX, engine="openpyxl") as writer:
        stops_no_access[["stop_id", "stop_name", "station_id", "tod_tier"]].to_excel(
            writer, sheet_name="Stops Without Access", index=False
        )
        access_no_stops[["access_id", "access_point_name", "station_id", "station_name"]].to_excel(
            writer, sheet_name="Access Without Stops", index=False
        )
    print(f"Linkage gaps exported → {LINKAGE_GAPS_XLSX}")
    raise ValueError(
        f"Station linkage gaps detected — {len(stops_no_access)} stop(s) with no access points, "
        f"{len(access_no_stops)} access point(s) with no stops. "
        f"Fix in the source data and re-run Step 3.\n"
        f"Details: {LINKAGE_GAPS_XLSX}"
    )

print("✓  All stops have at least one access point and all access points have at least one stop")
print(f"   {tod_merged_gdf['_merge'].eq('both').sum():,} matched stop/access-point pairs")


Access points with unmatched station_id:


,access_id,station_id,access_point_name,geometry,station_name,_merge


✓  All stops have at least one access point and all access points have at least one stop
   2,381 matched stop/access-point pairs


In [6]:
# Identify access points with tod_tier conflicts —
# i.e. a single access_id is associated with stops of more than one tod_tier.

tiers_per_access = tod_merged_gdf.groupby("access_id")["tod_tier"].nunique()
tier_conflict_access_ids = tiers_per_access[tiers_per_access > 1].index

print(f"{len(tier_conflict_access_ids):,} access points have tod_tier conflicts")

conflict_mask = tod_merged_gdf["access_id"].isin(tier_conflict_access_ids)
display_cols = ["access_id", "station_id", "station_name", "stop_id", "stop_name", "route_short_name", "route_type", "tod_tier"]

tod_tier_conflicts = (
    tod_merged_gdf[conflict_mask][display_cols]
    .sort_values(["access_id", "tod_tier"])
)

102 access points have tod_tier conflicts


In [7]:
# Resolve tod_tier conflicts by applying tier precedence: Tier 1 > Tier 2
# For each access point, retain only the highest-priority tier.
# Drop all stop-level fields; the result is one row per access point.

TIER_ORDER = ["Tier 1", "Tier 2"]

tod_access_pts_resolved_gdf = (
    tod_merged_gdf
    .assign(tod_tier=pd.Categorical(tod_merged_gdf["tod_tier"], categories=TIER_ORDER, ordered=True))
    .sort_values(["access_id", "tod_tier"])
    .drop_duplicates(subset="access_id", keep="first")
    [["access_id", "station_id", "access_point_name", "station_name", "tod_tier", "geometry"]]
    .copy()
)

# Restore tod_tier to plain string after deduplication.
tod_access_pts_resolved_gdf["tod_tier"] = tod_access_pts_resolved_gdf["tod_tier"].astype(str)

print(f"{len(tod_access_pts_resolved_gdf):,} access points ready for buffering")
print("Tier distribution:")
print(tod_access_pts_resolved_gdf["tod_tier"].value_counts().sort_index())
tod_access_pts_resolved_gdf


1,159 access points ready for buffering
Tier distribution:
tod_tier
Tier 1    421
Tier 2    738
Name: count, dtype: int64


,access_id,station_id,access_point_name,station_name,tod_tier,geometry
625,12TH_1,mtc:12th-st-oakland-city-center-bart,B2 13th St and Broadway Entrance / Exit,12th St Oakland City Center BART,Tier 1,POINT (-122.27172 37.80328)
622,12TH_2,mtc:12th-st-oakland-city-center-bart,A2 13th St and Broadway Entrance / Exit,12th St Oakland City Center BART,Tier 1,POINT (-122.27195 37.80337)
631,12TH_3,mtc:12th-st-oakland-city-center-bart,B3 13th St and Broadway Entrance / Exit,12th St Oakland City Center BART,Tier 1,POINT (-122.27133 37.80391)
628,12TH_4,mtc:12th-st-oakland-city-center-bart,A3 Oakland City Center Entrance / Exit,12th St Oakland City Center BART,Tier 1,POINT (-122.27185 37.80383)
640,12TH_5,mtc:12th-st-oakland-city-center-bart,A1 12th St and Broadway Entrance / Exit,12th St Oakland City Center BART,Tier 1,POINT (-122.27249 37.80248)
...,...,...,...,...,...,...
1122,mtc:1643701439649,mtc:fruitvale,Northbound Elevator (Platform Level),Fruitvale,Tier 1,POINT (-122.2242 37.77504)
1116,mtc:BA-FTVL_2-1643433662776,mtc:fruitvale,Enter/Exit : Station entrance,Fruitvale,Tier 1,POINT (-122.22445 37.77508)
1118,mtc:BA-FTVL_3-1643433667504,mtc:fruitvale,Enter/Exit : Station entrance,Fruitvale,Tier 1,POINT (-122.22448 37.77505)
2345,south_sf,south_sf,South San Francisco Caltrain Station,South San Francisco Caltrain Station,Tier 1,POINT (-122.40535 37.65598)


In [8]:
# Generate simple full-circle buffers for each access point at each distance band.
# No ring differencing is applied — overlaps between access points, tiers, and
# bands are resolved downstream using tod_tier, buffer_band, and geometry.
#
# Band definitions (full circles from access point):
#   200ft        :   60.96 m  — inner zone
#   quarter_mile :  402.336 m — ¼ mile
#   half_mile    :  804.672 m — ½ mile

BUFFER_200FT     =  60.96   # meters
BUFFER_QTR_MILE  = 402.336  # meters
BUFFER_HALF_MILE = 804.672  # meters

BUFFER_BANDS = [
    ("200ft",        BUFFER_200FT),
    ("quarter_mile", BUFFER_QTR_MILE),
    ("half_mile",    BUFFER_HALF_MILE),
]

# Project to EPSG:26910 (UTM Zone 10N) for accurate metric distances.
tod_access_pts_proj = tod_access_pts_resolved_gdf.to_crs("EPSG:26910")

# Build one buffered copy per band, tagged with buffer_band label.
band_gdfs = []
for band_label, distance_m in BUFFER_BANDS:
    band_gdf = tod_access_pts_proj.copy()
    band_gdf["geometry"] = tod_access_pts_proj.geometry.buffer(distance_m)
    band_gdf["buffer_band"] = band_label
    band_gdfs.append(band_gdf)

# Stack all bands — remain in EPSG:26910 for downstream processing.
tod_zone_buffers_gdf = gpd.GeoDataFrame(
    pd.concat(band_gdfs, ignore_index=True),
    crs="EPSG:26910",
)

# Verify row count.
expected_rows = len(tod_access_pts_resolved_gdf) * len(BUFFER_BANDS)
assert len(tod_zone_buffers_gdf) == expected_rows, (
    f"Expected {expected_rows} rows, got {len(tod_zone_buffers_gdf)}"
)

print(f"{len(tod_zone_buffers_gdf):,} total buffer features ({len(tod_access_pts_resolved_gdf):,} access points × {len(BUFFER_BANDS)} bands)")
print("\nRows per band:")
print(tod_zone_buffers_gdf["buffer_band"].value_counts())
print("\nRows per tier:")
print(tod_zone_buffers_gdf["tod_tier"].value_counts())

3,477 total buffer features (1,159 access points × 3 bands)

Rows per band:
buffer_band
200ft           1159
quarter_mile    1159
half_mile       1159
Name: count, dtype: int64

Rows per tier:
tod_tier
Tier 2    2214
Tier 1    1263
Name: count, dtype: int64


In [9]:
jurisdictions_with_pop

,objectid,fipst,fipco,jurisdiction,Shape__Area,Shape__Length,geometry,total_population
0,219,06,097,Unincorporated Sonoma,0.450046,8.231250,"MULTIPOLYGON (((-122.62741 38.66751, -122.6238...",NaN
1,220,06,041,Unincorporated Marin,0.196976,5.260678,"MULTIPOLYGON (((-122.34747 38.07327, -122.3516...",NaN
2,221,06,055,Unincorporated Napa,0.201949,4.566464,"MULTIPOLYGON (((-122.10329 38.51335, -122.1033...",NaN
3,222,06,095,Unincorporated Solano,0.200921,6.225325,"MULTIPOLYGON (((-121.66836 38.17504, -121.6708...",NaN
4,223,06,013,Unincorporated Contra Costa,0.128281,8.168448,"MULTIPOLYGON (((-121.55701 37.81649, -121.5578...",NaN
...,...,...,...,...,...,...,...,...
104,323,06,081,San Mateo,0.004186,0.465963,"POLYGON ((-122.27668 37.53423, -122.27673 37.5...",103555.0
105,324,06,081,South San Francisco,0.007979,0.701870,"POLYGON ((-122.39193 37.67177, -122.39134 37.6...",64487.0
106,325,06,081,Woodside,0.003023,0.481513,"POLYGON ((-122.27071 37.45591, -122.2711 37.45...",5181.0
107,326,06,013,El Cerrito,0.000978,0.164531,"POLYGON ((-122.31972 37.93834, -122.31957 37.9...",25781.0


In [10]:
# Erase jurisdictions under the 35k population threshold from half-mile buffers.
# All operations remain in EPSG:26910 (UTM Zone 10N) throughout.

# Minimum area (sq meters) for retaining polygon parts after the difference
# operation. Area analysis (see cell below) shows a hard gap: all slivers are
# essentially 0 m² (floating-point artifacts from the overlay), and even the
# smallest legitimate fragment is >1,279,000 m². Any value from 1 to ~1,279,000
# drops exactly the same artifacts. 100 m² is used as a conservative, legible default.
MIN_SLIVER_AREA_SQM = 100

unincorporated_counties = [
    "Unincorporated Alameda",
    "Unincorporated Contra Costa",
    "Unincorporated Marin",
    "Unincorporated Napa",
    "Unincorporated San Francisco",
    "Unincorporated San Mateo",
    "Unincorporated Santa Clara",
    "Unincorporated Solano",
    "Unincorporated Sonoma",
]

juris_under_thres = jurisdictions_with_pop[
    (jurisdictions_with_pop["jurisdiction"].isin(unincorporated_counties)) |
    (jurisdictions_with_pop["total_population"] < 35000)
].copy()

# Explode to ensure proper erasing of multi-part geometries.
juris_under_thres_exploded = juris_under_thres.explode(index_parts=False)

# Filter to just the half-mile band (already in EPSG:26910).
half_mile_buffers_gdf = tod_zone_buffers_gdf[
    tod_zone_buffers_gdf["buffer_band"] == "half_mile"
].copy()

# Project jurisdiction boundaries to EPSG:26910 for the overlay.
juris_under_thres_proj = juris_under_thres_exploded.to_crs("EPSG:26910")

# Perform the erase (both inputs in EPSG:26910, area filter is in sq meters).
half_mile_erased = gpd.overlay(
    df1=half_mile_buffers_gdf, df2=juris_under_thres_proj, how="difference"
)

# Explode multipolygons and drop zero-area floating-point artifacts produced
# by the overlay. See area analysis cell for threshold justification.
n_before = len(half_mile_erased)
half_mile_erased = (
    half_mile_erased
    .explode(index_parts=False)
    .loc[lambda gdf: gdf.geometry.area >= MIN_SLIVER_AREA_SQM]
)
n_dropped = n_before - len(half_mile_erased)
print(f"Sliver removal: {n_dropped:,} fragment(s) dropped (< {MIN_SLIVER_AREA_SQM} m²)")

tod_zone_buffers_final_gdf = gpd.GeoDataFrame(
    pd.concat(
        [
            tod_zone_buffers_gdf[tod_zone_buffers_gdf["buffer_band"] != "half_mile"],
            half_mile_erased,
        ],
        ignore_index=True,
    ),
    crs="EPSG:26910",
)

Sliver removal: 22 fragment(s) dropped (< 100 m²)


In [ ]:
# ── Priority Resolution — Step A: Split into per-zone GDFs and validate ──────
#
# Geographic precedence order (highest → lowest priority):
#   1. Tier 1 – 200ft
#   2. Tier 2 – 200ft
#   3. Tier 1 – Quarter Mile
#   4. Tier 1 – Half Mile
#   5. Tier 2 – Quarter Mile
#   6. Tier 2 – Half Mile
#
# Tier 1 takes full precedence over Tier 2 at all distance bands, with the
# exception that both 200ft zones are resolved first (Tier 2 200ft carries
# higher development standards than Tier 1 quarter-mile or half-mile).
# Sequential erase ensures each lower-priority zone contains no area already
# claimed by any higher-priority zone, producing 6 clean, non-overlapping layers.

ZONE_PRIORITY = [
    ("Tier 1", "200ft", "Tier 1 - 200ft"),
    ("Tier 2", "200ft", "Tier 2 - 200ft"),
    ("Tier 1", "quarter_mile", "Tier 1 - Quarter Mile"),
    ("Tier 1", "half_mile", "Tier 1 - Half Mile"),
    ("Tier 2", "quarter_mile", "Tier 2 - Quarter Mile"),
    ("Tier 2", "half_mile", "Tier 2 - Half Mile"),
]

# Split into one GDF per (tier, band) combination and repair geometry.
zone_gdfs = {}
for tod_tier, buffer_band, zone_label in ZONE_PRIORITY:
    mask = (tod_zone_buffers_final_gdf["tod_tier"] == tod_tier) & (
        tod_zone_buffers_final_gdf["buffer_band"] == buffer_band
    )
    gdf = tod_zone_buffers_final_gdf[mask].copy()
    gdf = repair_geometry(gdf)
    # repair_geometry can produce GeometryCollections — keep only polygonal types.
    gdf = gdf[gdf.geometry.geom_type.isin(["Polygon", "MultiPolygon"])].copy()
    zone_gdfs[zone_label] = gdf

print("Zone feature counts after split and repair:")
for zone_label, gdf in zone_gdfs.items():
    print(f"  {zone_label:<28} {len(gdf):>5} features  CRS: {gdf.crs.to_epsg()}")


Geodataframe contains valid geometry. No repair necessary.
Geodataframe contains valid geometry. No repair necessary.
Geodataframe contains valid geometry. No repair necessary.
Geodataframe contains valid geometry. No repair necessary.
Geodataframe contains valid geometry. No repair necessary.
Geodataframe contains valid geometry. No repair necessary.
Zone feature counts after split and repair:
  Tier 1 - 200ft                 421 features  CRS: 26910
  Tier 2 - 200ft                 738 features  CRS: 26910
  Tier 1 - Quarter Mile          421 features  CRS: 26910
  Tier 1 - Half Mile             378 features  CRS: 26910
  Tier 2 - Quarter Mile          738 features  CRS: 26910
  Tier 2 - Half Mile             738 features  CRS: 26910


In [12]:
# ── Priority Resolution — Step B: Sequential erase loop ──────────────────────
#
# For each zone (in priority order), erase all previously claimed geometry
# from the current zone. The accumulated mask grows with each iteration.

processed = []
accumulated_mask = None  # GeoDataFrame of claimed geometry (EPSG:26910)

for i, (tod_tier, buffer_band, zone_label) in enumerate(ZONE_PRIORITY):
    gdf = zone_gdfs[zone_label].copy()

    if i == 0:
        # Highest priority — kept unchanged.
        gdf["zone_label"] = zone_label
        processed.append(gdf)
        accumulated_mask = gdf[["geometry"]].copy()
        print(f"  {zone_label:<28} {len(gdf):>5} features  (priority 1, no erase)")
    else:
        # Erase all higher-priority geometry from this zone.
        mask_dissolved = accumulated_mask.dissolve()
        erased = gpd.overlay(gdf, mask_dissolved, how="difference")

        # Drop slivers produced by the overlay.
        n_before = len(erased)
        erased = (
            erased.explode(index_parts=False)
            .loc[lambda df: df.geometry.area >= MIN_SLIVER_AREA_SQM]
            .copy()
        )
        n_dropped = n_before - len(erased)

        # Repair geometry and drop any non-polygonal artifacts.
        erased = repair_geometry(erased)
        erased = erased[erased.geometry.geom_type.isin(["Polygon", "MultiPolygon"])].copy()

        erased["zone_label"] = zone_label
        processed.append(erased)

        # Grow the accumulated mask with only the geometry that was actually claimed.
        accumulated_mask = gpd.GeoDataFrame(
            pd.concat([accumulated_mask, erased[["geometry"]]], ignore_index=True),
            crs="EPSG:26910",
        )
        print(f"  {zone_label:<28} {len(erased):>5} features  ({n_dropped} slivers dropped)")

print(f"\nSequential erase complete. {len(processed)} zones processed.")

  Tier 1 - 200ft                 421 features  (priority 1, no erase)
Geodataframe contains valid geometry. No repair necessary.
  Tier 2 - 200ft                 731 features  (6 slivers dropped)
Geodataframe contains valid geometry. No repair necessary.
  Tier 1 - Quarter Mile          426 features  (-5 slivers dropped)
Geodataframe contains valid geometry. No repair necessary.
  Tier 1 - Half Mile             521 features  (-143 slivers dropped)
Geodataframe contains valid geometry. No repair necessary.
  Tier 2 - Quarter Mile          609 features  (28 slivers dropped)
Geodataframe contains valid geometry. No repair necessary.
  Tier 2 - Half Mile            1098 features  (-406 slivers dropped)

Sequential erase complete. 6 zones processed.


In [13]:
# ── Priority Resolution — Step C: Concat, reproject, and finalize schema ─────

tod_zones_gdf = gpd.GeoDataFrame(
    pd.concat(processed, ignore_index=True),
    crs="EPSG:26910",
)

# Dissolve by zone_label to merge all polygons of the same zone
# into a single (potentially multi-part) geometry.
tod_zones_gdf = tod_zones_gdf.dissolve(by="zone_label").reset_index()

# Final geometry repair and type filter.
tod_zones_gdf = repair_geometry(tod_zones_gdf)
tod_zones_gdf = tod_zones_gdf[
    tod_zones_gdf.geometry.geom_type.isin(["Polygon", "MultiPolygon"])
].copy()

# Reproject to EPSG:4326 for export.
tod_zones_gdf = tod_zones_gdf.to_crs("EPSG:4326")

# Assign a UUID4 zone_id to each polygon and select final schema columns.
# UUID4 is used rather than a geometry-based ID because post-erase polygons
# are irregular shapes (crescents, arcs) whose centroids can fall outside the
# polygon, making coordinate-based IDs unreliable.
tod_zones_gdf["zone_id"] = [str(uuid.uuid4()) for _ in range(len(tod_zones_gdf))]
tod_zones_gdf = tod_zones_gdf[["zone_id", "zone_label", "geometry"]].copy()

print(f"Total resolved zone features: {len(tod_zones_gdf):,}")
print(f"Columns: {list(tod_zones_gdf.columns)}")
print("\nFeature count by zone_label (priority order):")
for _, _, zone_label in ZONE_PRIORITY:
    n = (tod_zones_gdf["zone_label"] == zone_label).sum()
    print(f"  {zone_label:<28} {n:>5}")

Geodataframe contains valid geometry. No repair necessary.
Total resolved zone features: 6
Columns: ['zone_id', 'zone_label', 'geometry']

Feature count by zone_label (priority order):
  Tier 1 - 200ft                   1
  Tier 2 - 200ft                   1
  Tier 1 - Quarter Mile            1
  Tier 1 - Half Mile               1
  Tier 2 - Quarter Mile            1
  Tier 2 - Half Mile               1


In [14]:
# Map — Resolved TOD zones: colored by zone_label with built-in legend.
import tempfile, webbrowser
import folium

ZONE_COLORS = {
    "Tier 1 - 200ft":        "#006d2c",  # dark green
    "Tier 2 - 200ft":        "#31a354",  # medium green
    "Tier 1 - Quarter Mile": "#a63603",  # dark orange
    "Tier 2 - Quarter Mile": "#fd8d3c",  # light orange
    "Tier 1 - Half Mile":    "#08306b",  # dark blue
    "Tier 2 - Half Mile":    "#2171b5",  # medium blue
}

zone_labels = [label for _, _, label in ZONE_PRIORITY]

m2 = tod_zones_gdf.explore(
    column="zone_label",
    categories=zone_labels,
    cmap=[ZONE_COLORS[label] for label in zone_labels],
    tooltip=["zone_label"],
    style_kwds={"fillOpacity": 0.4, "weight": 1},
    legend=True,
    name="TOD Zones",
)

tod_access_pts_resolved_gdf.explore(
    m=m2,
    color="black",
    marker_kwds={"radius": 2, "fill": True, "fillColor": "black", "fillOpacity": 1},
    tooltip=["access_id", "access_point_name", "tod_tier"],
    name="access points",
)

jurisdictions_with_pop.explore(
    m=m2,
    tooltip=["jurisdiction", "total_population"],
    style_kwds={"fillOpacity": 0, "weight": 1.5, "color": "#555555"},
    name="jurisdictions",
)

folium.LayerControl(collapsed=False).add_to(m2)

tmp2 = tempfile.NamedTemporaryFile(suffix=".html", delete=False)
m2.save(tmp2.name)
webbrowser.open(f"file://{tmp2.name}")
print(f"Map saved to: {tmp2.name}")


Map saved to: /var/folders/9q/xt2lctm54xq6fd45m1lmgp4m0000gp/T/tmpkhu9jml8.html


In [15]:
# Export raw per-access-point buffers to the shared GeoPackage.
# Full circles at each distance band — one record per access point per band.
# Fields: access_id, station_id, access_point_name, station_name, tod_tier, buffer_band, geometry

tod_zone_buffers_final_gdf.to_crs("EPSG:4326").to_file(
    TOD_DATABASE_GPKG, layer=GPKG_TOD_ZONE_BUFFERS_LAYER, driver="GPKG", mode="w"
)

# Export jurisdiction boundaries with population attributes for reference.
jurisdictions_with_pop.to_file(
    TOD_DATABASE_GPKG, layer=GPKG_JURISDICTIONS_WITH_POP_LAYER, driver="GPKG", mode="w"
)

# Export resolved TOD zones to the shared GeoPackage.
# One record per non-overlapping zone polygon. Field: zone_label, geometry.

tod_zones_gdf.to_file(
    TOD_DATABASE_GPKG, layer=GPKG_TOD_ZONES_LAYER, driver="GPKG", mode="w"
)

print(f"Exported {len(tod_zone_buffers_final_gdf):,} buffer features → '{GPKG_TOD_ZONE_BUFFERS_LAYER}'")
print(f"Exported {len(jurisdictions_with_pop):,} jurisdiction features → '{GPKG_JURISDICTIONS_WITH_POP_LAYER}'")
print(f"\nOutput: {TOD_DATABASE_GPKG}")
print(f"\nBuffer features by tier and band:")
print(tod_zone_buffers_final_gdf.groupby(["tod_tier", "buffer_band"]).size().rename("count").to_string())
print(f"Exported {len(tod_zones_gdf):,} resolved zone features → '{GPKG_TOD_ZONES_LAYER}'")
print(f"Output: {TOD_DATABASE_GPKG}")

Exported 3,434 buffer features → 'tod_zone_buffers'
Exported 109 jurisdiction features → 'jurisdictions_with_pop_acs2019_2023'

Output: /Users/jcroff/Library/CloudStorage/Box-Box/DSA Projects/Spatial Analysis and Mapping/SB79 Transit Oriented Development/Data/SB79_tod_database.gpkg

Buffer features by tier and band:
tod_tier  buffer_band 
Tier 1    200ft           421
          half_mile       378
          quarter_mile    421
Tier 2    200ft           738
          half_mile       738
          quarter_mile    738
Exported 6 resolved zone features → 'tod_zones'
Output: /Users/jcroff/Library/CloudStorage/Box-Box/DSA Projects/Spatial Analysis and Mapping/SB79 Transit Oriented Development/Data/SB79_tod_database.gpkg
